In [ ]:
# Cell 0: synchronize the repository environment before running the workbench.
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Find the workspace root without confusing member pyproject files for it."""
    for candidate in (start, *start.parents):
        if (
            (candidate / "uv.lock").exists()
            and (candidate / "implementations").is_dir()
            and (candidate / "aieng-forecasting").is_dir()
        ):
            return candidate
    raise FileNotFoundError("Could not find the agentic-forecasting repository root.")


SETUP_ROOT = find_repo_root(Path.cwd().resolve())
print("Running: uv sync --all-extras --dev --all-packages")
setup_result = subprocess.run(
    ["uv", "sync", "--all-extras", "--dev", "--all-packages"],
    cwd=SETUP_ROOT,
    check=False,
    text=True,
    capture_output=True,
)
print(setup_result.stdout)
if setup_result.returncode != 0:
    print(setup_result.stderr)
    raise RuntimeError("uv sync failed. Install uv or restart the notebook with the repository environment selected.")
print("Environment synchronized successfully.")
print("If packages changed, restart the notebook kernel before continuing.")

# Manufacturing Stress Parameter Sweep Workbench

This notebook runs the same deterministic workflow as `run_parameter_smoke.py`, directly through its Python functions. It makes no LLM calls and does not use prediction-result caches.

The workflow deliberately has two stages:

1. **Tune** all candidates over 2000–2017.
2. **Confirm** only the selected tuning winner over 2018–2024.

Do not use the confirmation window to choose a model; doing so would turn it into another tuning set.


In [ ]:
# Main controls: change these values, then run the notebook from top to bottom.
STAGE = "tune"
CANDIDATE = None
BACKTEST_STRIDE = 3
REFRESH_INPUT_DATA = False
RUN_SWEEP = True

# For confirmation, set STAGE = "confirm" and CANDIDATE to the winner
# printed by the tuning stage. Keep BACKTEST_STRIDE unchanged.
print(
    {
        "stage": STAGE,
        "candidate": CANDIDATE,
        "backtest_stride": BACKTEST_STRIDE,
        "refresh_input_data": REFRESH_INPUT_DATA,
        "run_sweep": RUN_SWEEP,
    }
)

## Imports and active features

The notebook imports the candidate definitions and evaluation helpers from `run_parameter_smoke.py`; it does not duplicate the sweep logic. The active feature list is printed so changes in `features.py` are visible before the run starts.


In [ ]:
import yaml
from aieng.forecasting.evaluation import BacktestSpec
from dotenv import load_dotenv
from IPython.display import display as ipython_display
from manufacturing_stress_forecasting.data import build_manufacturing_stress_service
from manufacturing_stress_forecasting.features import STATISTICAL_FEATURE_SERIES_IDS
from manufacturing_stress_forecasting.run_parameter_smoke import (
    BASELINE_NAME,
    CANDIDATES,
    build_experiment_spec,
    comparison_table,
    predictors_for_stage,
    run_candidates,
)


REPO_ROOT = SETUP_ROOT
load_dotenv(REPO_ROOT / ".env", override=False)
SPEC_PATH = (
    REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "specs" / "manufacturing_stress_smoke.yaml"
)
print(f"Repository: {REPO_ROOT}")
print(f"Active statistical features ({len(STATISTICAL_FEATURE_SERIES_IDS)}):")
for feature_name in STATISTICAL_FEATURE_SERIES_IDS:
    print(f"  - {feature_name}")

## Create or refresh the data service

The default reads the existing local FRED and New York Fed caches. Set `REFRESH_INPUT_DATA = True` only when you intentionally want to download fresh source data.


In [ ]:
service = build_manufacturing_stress_service(
    cache_dir=REPO_ROOT / "data" / "fred",
    gscpi_cache_path=REPO_ROOT / "data" / "new_york_fed" / "gscpi_interactive_data.csv",
    refresh=REFRESH_INPUT_DATA,
)
ipython_display(service.summary())

## Build the tuning or confirmation experiment

The stage determines the date window and candidate set. Tuning runs the historical-frequency baseline plus every configured candidate. Confirmation runs the baseline plus exactly one selected candidate.


In [ ]:
if STAGE not in {"tune", "confirm"}:
    raise ValueError("STAGE must be 'tune' or 'confirm'.")
if BACKTEST_STRIDE not in {1, 3}:
    raise ValueError("BACKTEST_STRIDE must be 1 or 3.")
if STAGE == "confirm" and CANDIDATE is None:
    raise ValueError("Set CANDIDATE to the tuning winner before confirmation.")
if STAGE == "tune" and CANDIDATE is not None:
    raise ValueError("Leave CANDIDATE as None during tuning.")

with SPEC_PATH.open() as file:
    base_spec = BacktestSpec.model_validate(yaml.safe_load(file))

spec = build_experiment_spec(base_spec, stage=STAGE, stride=BACKTEST_STRIDE)
named_predictors = predictors_for_stage(STAGE, CANDIDATE)

print(
    f"Stage={STAGE}; origins={spec.start.date()} to {spec.end.date()}; "
    f"stride={spec.stride}; horizon={spec.task.horizons[0]} month(s)"
)
print("Predictors:")
for predictor_name, _predictor in named_predictors:
    print(f"  - {predictor_name}")

if STAGE == "tune":
    print("Available confirmation names:")
    for candidate in CANDIDATES:
        print(f"  - {candidate.name}")

## Run the parameter sweep

This can take several minutes because every predictor is refit at every forecast origin. Set `RUN_SWEEP = False` when you only want to inspect the setup.


In [ ]:
results = None
table = None
if RUN_SWEEP:
    results = run_candidates(named_predictors, spec=spec, service=service)
    table = comparison_table(results)
else:
    print("Sweep skipped because RUN_SWEEP is False.")

## Compare Brier scores

Lower `mean_brier` is better. A negative `delta_vs_baseline` and positive `brier_skill` indicate improvement over historical frequency. `scored` and `skipped` verify that the comparison used the expected forecast origins.


In [ ]:
if table is not None:
    ipython_display(
        table.style.format(
            {
                "mean_brier": "{:.4f}",
                "delta_vs_baseline": "{:+.4f}",
                "brier_skill": "{:+.1%}",
            }
        )
    )

    if STAGE == "tune":
        best_model = table[table["candidate"] != BASELINE_NAME].iloc[0]
        selected_candidate = str(best_model["candidate"])
        print(f"Best non-baseline tuning candidate: {selected_candidate}")
        if float(best_model["brier_skill"]) <= 0:
            print("Historical frequency still has the lower tuning-period Brier score.")
        print("To confirm it, set the controls to:")
        print('STAGE = "confirm"')
        print(f'CANDIDATE = "{selected_candidate}"')
        print(f"BACKTEST_STRIDE = {BACKTEST_STRIDE}")
    else:
        print(f"Confirmation completed for {CANDIDATE}.")

## Confirmation discipline

After the tuning run, copy the printed winner into `CANDIDATE`, change `STAGE` to `"confirm"`, keep the same stride, and rerun the notebook. Confirmation deliberately evaluates only that candidate and historical frequency. If you inspect multiple candidates on 2018–2024 and then choose among them, the period is no longer a protected confirmation window.
